<a href="https://colab.research.google.com/github/zia207/Causal_Inference_R/blob/main/Notebook/02_08_05_00_casual_ml_introduction_r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![R Banner](https://drive.google.com/uc?id=1sRkZDyR2Ss9-2aQuSseKOM2lP7RVH70C)

# 5. Causal Machine Learning

Causal inference and machine learning are two interconnected fields within data science and statistics that address how we extract meaningful insights from data, though they emphasize different aspects of understanding relationships and making decisions. Causal inference focuses on determining cause-and-effect relationships, answering questions like 'Does A cause B?' or 'What would happen if we changed A?' Machine learning, on the other hand, excels at pattern recognition and prediction, often dealing with associations rather than direct causation. Their relationship has grown increasingly symbiotic in recent years, with machine learning techniques enhancing causal inference methods and vice versa, leading to more robust, real-world applications in areas such as healthcare, economics, and policy-making.

To start with causal inference, it is fundamentally about going beyond mere correlations in data to establish causality. In everyday data analysis, we might observe that two variables move together — such as higher ice cream sales coinciding with increased drowning incidents — but correlation does not imply causation; here, summer heat is the true driver of both. Causal inference provides tools to disentangle such confounders and identify true causal effects. Pioneered by statisticians like Donald Rubin and Judea Pearl, it relies on frameworks such as potential outcomes (considering counterfactuals, or what would have happened without an intervention) and causal graphs (directed acyclic graphs that map out variables and their directional influences). Key methods include randomized controlled trials (RCTs), which are the gold standard for establishing causality by randomly assigning treatments to eliminate biases; instrumental variables, which use external factors that affect the treatment but not the outcome directly; and propensity score matching, which balances groups in observational data to mimic randomization. The goal is to estimate causal effects reliably, often in scenarios where experiments are unethical or impractical, like assessing the impact of smoking on health without forcing people to smoke.

Machine learning, by contrast, is a subset of artificial intelligence that involves training algorithms on data to make predictions or decisions without being explicitly programmed for the task. It includes supervised learning (e.g., regression or classification models like random forests or neural networks that learn from labeled data), unsupervised learning (e.g., clustering or dimensionality reduction to find hidden patterns), and reinforcement learning (where agents learn optimal actions through trial and error). ML models are incredibly powerful for tasks like image recognition, recommendation systems, or forecasting stock prices, as they can handle vast, high-dimensional datasets and uncover complex nonlinear relationships. However, traditional ML is primarily associative: it predicts outcomes based on patterns in training data but does not inherently distinguish between correlation and causation. For instance, an ML model might predict higher lung cancer rates among coffee drinkers, but it would not explain whether coffee causes the cancer or if smoking is the confounding factor.

The relationship between causal inference and machine learning has evolved into a fertile area known as "causal machine learning" or "causal AI." This integration addresses the limitations of each field alone. Machine learning's strength in handling big data and complex patterns can supercharge causal inference by automating tasks like confounder adjustment or estimating heterogeneous treatment effects across subgroups. For example, techniques like double/debiased machine learning use ML models to estimate nuisance parameters (such as propensity scores) more accurately than traditional parametric methods, leading to better causal estimates in high-dimensional settings. Causal forests extend random forests to estimate individualized treatment effects, allowing for personalized predictions like "How much would this patient's blood pressure drop if they took this drug?" Conversely, incorporating causal inference into ML makes models more interpretable and robust to distribution shifts — changes in data patterns over time or across contexts. In reinforcement learning, causal structures help agents understand interventions rather than just observing states, improving decision-making in dynamic environments like autonomous driving or robotics.

This synergy is particularly valuable in applied domains. In healthcare, causal ML can evaluate treatment efficacy from electronic health records, accounting for biases in who receives which drug. In economics, it helps assess policy impacts, such as the causal effect of minimum wage increases on employment. Even in tech, companies like Google or Meta use causal inference to optimize A/B tests with ML, ensuring that algorithmic changes truly cause improvements in user engagement rather than just correlating with them. Challenges remain, such as ensuring assumptions like "no unmeasured confounders" hold true, or dealing with the computational demands of combining these approaches, but ongoing research — bolstered by libraries like EconML, DoWhy, or CausalML — continues to bridge the gap.

In summary, while causal inference provides the theoretical backbone for understanding why things happen, machine learning offers the computational muscle to scale and apply those insights practically. Their interplay represents a shift toward more trustworthy AI systems that not only predict but also explain and intervene effectively, paving the way for advancements in evidence-based decision-making across disciplines.

## Setup R in Python Runtime - Install {rpy2}

{rpy2} allows running R code in Colab Python runtime via `%%R` magic.

In [ ]:
!pip uninstall rpy2 -y
!pip install rpy2==3.5.1
%load_ext rpy2.ipython

## Mount Google Drive

Mount Google Drive if your data or R library folder is stored there.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Check and Install Required R Packages

In [ ]:
%%R
packages <- c(
  "tidyverse",
  "grf",
  "DoubleML",
  "mlr3",
  "mlr3learners",
  "tmle",
  "SuperLearner",
  "WeightIt",
  "MatchIt",
  "cobalt",
  "lmtp",
  "hdm"
)

In [ ]:
%%R
new_packages <- packages[!(packages %in% installed.packages(lib = 'drive/My Drive/R/')[,"Package"])]
if (length(new_packages)) install.packages(new_packages, lib = 'drive/My Drive/R/')

### Verify Installation

In [ ]:
%%R
# set library path
.libPaths('drive/My Drive/R')
cat("Installed packages:\n")
print(sapply(packages, requireNamespace, quietly = TRUE))

### Load Packages

In [ ]:
%%R
invisible(lapply(packages, function(pkg) {
  suppressPackageStartupMessages(library(pkg, character.only = TRUE))
}))

### Check Loaded Packages

In [ ]:
%%R
cat("Successfully loaded packages:\n")
print(search()[grepl("package:", search())])

## Applications of Causal Inference in Machine Learning

Causal inference applies to machine learning by bridging the gap between **pure prediction** (what traditional ML excels at) and **understanding cause-and-effect relationships** (what enables reliable interventions, policy decisions, and robust generalizations). While standard machine learning models learn associations from data, they often fail when the underlying environment changes or when we need to answer "what if" questions about actions we have not observed. Causal inference brings rigorous tools — such as potential outcomes, directed acyclic graphs (DAGs), and identification strategies — into ML pipelines, creating the field known as **causal machine learning** (or causal ML). This integration allows ML to estimate treatment effects, handle confounding, and support decision-making in high-stakes domains like healthcare, economics, marketing, and personalized AI.

Below, we explore the main ways causal inference is applied in machine learning, organized under key sub-headings.

### Enhancing Causal Effect Estimation with Machine Learning

Traditional causal inference methods (e.g., propensity score matching, instrumental variables, or regression adjustment) often rely on simple parametric models, which can introduce bias when relationships are nonlinear or high-dimensional. Machine learning steps in to estimate these nuisance functions (like propensity scores or outcome regressions) more flexibly and accurately.

A flagship example is **Double/Debiased Machine Learning (Double ML)**, which uses any ML algorithm (random forests, gradient boosting, neural nets) to first estimate confounding relationships, then plugs them into an orthogonalized estimator for the causal parameter (e.g., average treatment effect, ATE). This "debiasing" step reduces sensitivity to model misspecification and enables reliable inference even in high-dimensional data. Libraries like EconML (from Microsoft) and DoubleML implement this approach widely for observational studies and A/B test augmentation.

### Estimating Heterogeneous and Individual Treatment Effects

One of the most powerful applications is moving from average effects to **conditional average treatment effects (CATE)** or **individual treatment effects (ITE)** — essentially, "how much does this treatment benefit person X specifically?"

**Causal forests** (and generalized random forests) extend random forests by splitting on heterogeneity in treatment effects rather than just outcome prediction. They provide honest estimates with confidence intervals and are particularly useful for personalized medicine or targeted marketing. Similarly, **meta-learners** (S-learner, T-learner, X-learner) combine multiple ML models to estimate CATE directly. These methods shine in uplift modeling, where businesses want to know which customers will respond positively to an intervention (e.g., a discount email), avoiding wasteful treatment of non-responders.

### Uplift Modeling and Personalized Interventions

Uplift modeling is a direct business-oriented application of causal ML, focused on estimating the incremental impact (uplift) of an action on an outcome. It answers: "Will this customer buy more if sent a promotion?"

Techniques include uplift trees/forests, which modify splitting criteria to maximize differences in treatment response, and meta-learners adapted for uplift. Packages like CausalML (from Uber) provide ready implementations for these, often used in e-commerce, digital advertising, and customer retention. By ranking users by predicted uplift, companies optimize resource allocation and improve ROI.

### Improving Robustness and Generalization in ML Models

Causal inference makes ML more robust to distribution shifts (covariate shift, concept drift) and interventions. By modeling causal structures via DAGs (using tools like DoWhy), models can simulate "what-if" scenarios and predict outcomes under hypothetical changes.

In reinforcement learning, causal world models help agents reason about interventions rather than just correlations, improving sample efficiency and transferability. Causal discovery algorithms combined with ML can even learn causal graphs from data, aiding interpretability in complex systems.

### Real-World Applications Across Domains

In healthcare, causal ML estimates personalized treatment effects from electronic records while controlling for confounding. In economics and policy, it evaluates interventions (e.g., job training programs) using observational big data. In tech platforms, it powers better A/B testing, recommendation systems that account for causal impact on engagement, and fair AI by detecting and mitigating biases through causal reasoning.

### Challenges and the Road Ahead

Despite the progress, challenges remain: strong assumptions (no unmeasured confounding, positivity), computational cost in very high dimensions, and the need for domain knowledge to specify causal models correctly. Ongoing research focuses on scalable methods, sensitivity analysis, and integration with deep learning.

In essence, causal inference transforms machine learning from a tool that predicts "what is" into one that reliably answers "what happens if we do this?" — a critical evolution for trustworthy, actionable AI in the real world.

## Popular Methods in Causal Machine Learning (CausalML/Causal AI)

Causal Machine Learning (**CausalML**) and **Causal AI** represent the exciting fusion of causal inference principles with modern machine learning techniques. These approaches go beyond traditional ML's focus on correlations and predictions, enabling systems to reason about cause-and-effect relationships, interventions ("what if we change X?"), counterfactuals ("what would have happened if...?"), and more robust decision-making.

CausalML/Causal AI encompasses a wide variety of methods, ranging from flexible estimation of treatment effects in high-dimensional data to full causal reasoning frameworks that support discovery, inference, and planning. Below are the major categories and popular methods, grouped logically.

### Meta-Learners for Treatment Effect Estimation

These "meta" approaches combine standard ML models (like random forests, gradient boosting, or neural nets) to estimate **Average Treatment Effect (ATE)**, **Conditional Average Treatment Effect (CATE)**, or **Individual Treatment Effect (ITE)**.

Popular variants include:

- **S-Learner** — Trains a single model on both treatment and features to predict outcomes.
- **T-Learner** — Fits separate models for treated and control groups, then subtracts predictions.
- **X-Learner** — Builds on T-Learner by cross-fitting and focusing on the difference between groups (often more efficient, especially with imbalanced treatment).
- **R-Learner** and **DR-Learner** (Doubly Robust) — Use outcome residuals and propensity scores for debiased estimation.

These are widely implemented in libraries like **CausalML** (from Uber) and **EconML** (from Microsoft).

### Tree-Based and Ensemble Methods for Heterogeneous Effects

These extend decision trees and forests to directly target causal heterogeneity (variation in treatment effects across individuals).

Key methods:

- **Causal Forests** (and Generalized Random Forests) — Adapted random forests that split based on treatment effect heterogeneity rather than just outcome prediction; provide honest estimates with confidence intervals.
- **Uplift Trees / Uplift Random Forests** — Designed specifically for uplift modeling (incremental impact of an action); use criteria like KL divergence, chi-square, or Euclidean distance for splits.
- **Causal Trees** — Single-tree versions focused on maximizing treatment effect differences.

These excel in personalized applications like marketing, medicine, and policy (e.g., "which customers will respond most to this promotion?").

### Double/Debiased Machine Learning (Double ML)

A cornerstone method for reliable causal estimation in high-dimensional or observational data.

- **Double/Debiased ML** — Uses ML to flexibly estimate nuisance functions (propensity scores and outcome models), then plugs them into an orthogonalized estimator to reduce bias and achieve \u221an consistency.
- Extensions include Bayesian variants and targeted maximum likelihood estimation (**TMLE**).

This family is particularly powerful for large-scale, real-world data where traditional parametric models fail.

### Deep Learning and Neural Network-Based Causal Methods

These integrate causality into deep architectures for complex, high-dimensional data (images, text, time series).

Examples:

- **Deep IV** (Deep Instrumental Variables) — Uses neural nets for nonlinear instrumental variable estimation.
- **Causal Representation Learning** — Learns disentangled, invariant representations (e.g., via contrastive or adversarial methods) to improve robustness and generalization.
- **Dragonnet / CEVAE** (Causal Effect Variational Autoencoder) — Variational approaches for counterfactual estimation.

These are increasingly used in computer vision, NLP, and reinforcement learning.

### Causal Discovery and Structural Causal Models (SCMs)

Methods that learn the underlying causal graph (Directed Acyclic Graph — DAG) from data, often combined with domain knowledge.

Popular algorithms:

- **PC algorithm** (Peter-Clark) — Constraint-based discovery using conditional independence tests.
- **GES / FGES** (Greedy Equivalence Search) — Score-based search for best-fitting DAG.
- **LiNGAM** — For linear non-Gaussian acyclic models.

Once discovered, SCMs enable interventions (do-calculus), counterfactuals, and root-cause analysis.

### Other Emerging and Specialized Approaches

- **Causal Reinforcement Learning** — Agents use causal models for better exploration, planning, and generalization in dynamic environments.
- **Causal Fairness and Explanations** — Methods that mitigate biases by modeling causal paths (e.g., removing spurious correlations).
- **Conformal Prediction for Causal Uncertainty** — Provides valid confidence intervals for causal estimates.

## Popular Open-Source Libraries and Frameworks

Many of these methods are readily available in production-ready packages:

- **CausalML** (Uber) — Focuses on uplift modeling, meta-learners, tree-based methods.

- **EconML** (Microsoft) — Double ML, causal forests, policy learning.

- **DoWhy** (Microsoft) — End-to-end causal reasoning with DAGs, identification, estimation, and refutation.

- **CausalNex / PyWhy** — Graph-based discovery and modeling.

- **DoubleML** (Python and R) — Robust double/debiased ML for causal inference.

- **grf** (R) — Generalized random forests for heterogeneous treatment effects.

- **skgrf** (Python) — Causal forests for Python.

- **RCausalML** (R) — A new package inspired by Python's CausalML and Microsoft's EconML, offering a wide range of causal inference methods including meta-learners, double ML, tree-based methods, deep learning approaches, and causal discovery algorithms.

- Others: Causal-learn, pgmpy (for probabilistic graphical models).

In practice, the choice of method depends on your data (experimental vs. observational), dimensionality, and goal (e.g., average effects vs. personalized uplift vs. full causal discovery). These techniques are transforming fields like healthcare (personalized treatments), economics (policy evaluation), marketing (targeted interventions), and AI safety (more robust, explainable systems). The field continues to evolve rapidly, with ongoing research addressing scalability, unmeasured confounding, and integration with large language models.

**Several excellent R packages implement methods for Causal Machine Learning (CausalML / Causal AI)**, focusing on heterogeneous treatment effects, uplift modeling, double/debiased machine learning, causal forests, and related techniques. While there is **no direct one-to-one equivalent** to the popular Python `causalml` package (from Uber, with strong emphasis on uplift trees/forests and meta-learners), the R ecosystem is very rich — especially for causal forests, double ML, and general causal inference with ML integration.

Here are the most important and actively used R packages for CausalML (as of early 2026):

### Generalized Random Forests (`grf`)

This is currently the **most popular and powerful package** for causal machine learning in R.

- Implements **causal forests** (and multi-arm variants) to estimate conditional average treatment effects (CATE / heterogeneous treatment effects).
- Supports instrumental variables, survival outcomes (causal survival forests), quantile treatment effects, and more.
- Provides honest splitting, confidence intervals, best linear projections (BLP), rank-weighted average treatment effects (RATE), and variable importance.
- Doubly robust estimation of average treatment effects (ATE).
- Actively maintained, excellent documentation, and widely used in both academia and industry.

→ Install: `install.packages("grf")`  
→ Website: https://grf-labs.github.io/grf/

### DoubleML

Excellent implementation of **Double/Debiased Machine Learning** (DML) for causal inference with high-dimensional data.

- Flexible nuisance estimation using any `mlr3` learner (random forests, xgboost, neural nets, etc.).
- Estimates ATE, ATT, local average treatment effects (LATE) with IV, and more.
- Very robust statistical inference (√n-consistent, valid CIs even with complex ML models).
- Companion to the popular Python `DoubleML` package.

→ Install: `install.packages("DoubleML")`  
→ Works best together with `mlr3` ecosystem.

### tmle / Targeted Learning ecosystem

Implements **Targeted Maximum Likelihood Estimation (TMLE)** — a doubly robust, efficient method that integrates machine learning (via Super Learner).

- Very strong theoretical foundations (van der Laan & Rose).
- Often combined with `SuperLearner` package for ensemble ML.
- Excellent for semi-parametric efficient estimation of causal effects.

→ Install: `install.packages("tmle")`

### RCausalML

[RCausalML](https://zia207.github.io/RCausalML/) is a new R package for causal inference in ML models, inspired by Python's CausalML and Microsoft's EconML. It features automated ML for nuisance models, policy learning, CATE interpretation, DR-learner family, and multiple neural causal models (CEVAE, DragonNet, GANITE, and more) with support for SCMs, DAG learning (NOTEARS, DAG-GNN, GraN-DAG), GNN and transformer-based causal models, and counterfactual analysis. Includes tools for causal structure learning and example notebooks. Most neural modules require the R torch package.

Key Features:

`Meta-Learners`  
Five meta-learners for estimating treatment effects:
- S-Learner: single-model approach
- T-Learner: treatment-specific models
- X-Learner: cross-fitting for improved accuracy in imbalanced settings
- R-Learner: residual-based approach for partially linear models
- DR-Learner: doubly robust estimation for extra protection against model misspecification

`Double Machine Learning (DML)`  
Offers Double Machine Learning (DML) for high-dimensional data, including several DML estimators and integration with DoubleML for advanced models.

`Tree-Based Methods`
- Uplift Random Forests and Causal Trees
- Causal Trees and Interaction Trees — Causal Survival Forest
- Causal XGBoost and Multi-Arm Causal Boosting — Multi-Arm Causal Boosting for K ≥ 2 treatment arms

`Deep Causal Machine Learning`  
Integrates deep learning methods for advanced users:
- Treatment effect estimators: TARNet, CFRNet, DragonNet, GANITE
- Generative models: CEVAE, CausalGAN, Deep Structural Causal Models
- Time-series causal discovery: Neural Granger Causality, RNNs, LSTMs, Graph Neural Networks, Transformer-based models
- Counterfactual reasoning: DeepSynth, CRN, G-Net
- NOTEARS (linear and nonlinear) — DAG-GNN (graph neural network approach)
- GraN-DAG (gradient-based neural DAG learner) — CASTLE
- SHAP Integration & Interpretability: Integrates with SHAP for interpretability and feature importance visualization.

→ Install: `install.packages("RCausalML")`

### Other Notable & Specialized Packages

- **WeightIt** + **cobalt** + **MatchIt** → Modern propensity score weighting/matching + balance diagnostics (great preprocessing for causal ML).
- **hdm** → High-dimensional methods (Lasso-based) with rigorous inference (often used as baseline or nuisance learner).
- **causaldrf** → Causal dose-response functions.
- **lmtp** → Longitudinal modified treatment policies with ML (very flexible for time-varying treatments).
- **ddml** → Another clean double/debiased ML implementation (simpler interface than DoubleML in some cases).

### Quick Comparison Table (Main CausalML-style Packages)

| Package | Main Strength | Heterogeneous Effects (CATE/ITE) | Double/Debiased ML | Uplift/Forest Style | Actively Maintained (2026) |
|------------|-------------|------------|------------|------------|------------|
| grf | Causal forests, very flexible & fast | ★★★★★ | ★★★ | ★★★★★ | Yes |
| DoubleML | Rigorous double ML with any learner | ★★★ | ★★★★★ | ★★ | Yes |
| tmle | Targeted ML, semi-parametric efficiency | ★★★ | ★★★★ | ★★ | Yes |
| lmtp | Longitudinal/time-varying treatments | ★★★ | ★★★ | ★★★ | Yes |

### Recommendation (2026 Perspective)

- Start with **grf** if you want something similar to Python's CausalML uplift forests / causal trees — it's the closest and most powerful all-rounder for heterogeneous effects.
- Use **DoubleML** when you need very robust inference in high-dimensional or complex settings (many covariates, nonlinear relationships).
- Combine them! Many workflows use `grf` or `mlr3` learners inside `DoubleML` for nuisance estimation.

## Summary and Conclusion

Causal inference and machine learning are two complementary fields that, when combined, create powerful tools for understanding and acting upon data. Causal inference provides the theoretical framework and methodologies to identify cause-and-effect relationships, while machine learning offers advanced algorithms capable of handling large, complex datasets. The integration of these fields, known as causal machine learning or causal AI, allows for more accurate estimation of treatment effects, personalized interventions, and robust decision-making in various domains such as healthcare, economics, and marketing. Popular methods in causal machine learning include meta-learners for treatment effect estimation, tree-based methods for heterogeneous effects, double/debiased machine learning, deep learning approaches, and causal discovery algorithms. R packages like `grf`, `DoubleML`, and `tmle` provide practical implementations of these methods, enabling researchers and practitioners to apply causal inference techniques effectively.

By leveraging the strengths of both causal inference and machine learning, we can move beyond mere predictions to actionable insights that inform policy, improve treatments, and optimize interventions. As research continues to advance in this area, we can expect even more sophisticated methods and applications that will further enhance our ability to make data-driven decisions with confidence in their causal validity.

## Resources

### Key Books

- **Applied Causal Inference Powered by ML and AI** by Victor Chernozhukov, Christian Hansen, Nathan Kallus, Martin Spindler, and Vasilis Syrgkanis (2024–2025, free online at causalml-book.org).  
  This is widely considered one of the best modern overviews, covering DAGs, structural causal models, double/debiased ML, and more, with intuition, technical details, and self-learning notebooks in R and Python.

- **Causal Inference: What If** by Miguel Hernán and Jamie Robins (free online draft).  
  The go-to practical foundation for core causal concepts (potential outcomes), with strong real-world applications.

- **Elements of Causal Inference: Foundations and Learning Algorithms** by Jonas Peters, Dominik Janzing, and Bernhard Schölkopf (MIT Press, free PDF available).  
  Excellent ML-focused bridge to causal discovery and algorithms.

### Top Online Courses and Tutorials

- **Machine Learning & Causal Inference: A Short Course** by Stanford Graduate School of Business (Golub Capital Social Impact Lab) — free video series on YouTube, plus an accompanying R Markdown tutorial/bookdown site with theory, code examples (heavy on causal forests via grf), and real data applications.  
  Great for hands-on learning.

- **Introduction to Causal Inference** by Brady Neal — free course with a draft textbook, strong on ML perspectives and heterogeneous effects.

### R-Specific Resources for CausalML Packages

- **grf (Generalized Random Forests)** — Primary package for causal forests and heterogeneous effects; check the official documentation and examples at grf-labs.github.io/grf for tutorials and vignettes.

- **DoubleML** — For rigorous double/debiased ML; excellent user guide, function reference, and reproducible examples at docs.doubleml.org (includes slides and code for various models).